In [1]:
import torch
import numpy as np

from dice import Dice, DiceFunctions
from functions import (
    MultiheadDiscreteSANetwork
)

In [2]:
DEVICE = 'cpu'
SEED = 784

In [3]:
states = torch.load('../../gym/mc/data/states.pt', weights_only=False)
actions = torch.load('../../gym/mc/data/actions.pt', weights_only=False)
rewards = torch.load('../../gym/mc/data/rewards.pt', weights_only=False)
dqn_actions = torch.load('../../gym/mc/data/dqn_actions.pt', weights_only=False)

In [4]:
state_dim = states[0].shape[1]
action_dim = 3
hidden_dim = 4

q_func = MultiheadDiscreteSANetwork(
    state_dim=state_dim,
    hidden_dim=hidden_dim,
    action_dim=action_dim,
    num_layers=2,
    seed=SEED
)

w_func = MultiheadDiscreteSANetwork(
    state_dim=state_dim,
    hidden_dim=hidden_dim,
    action_dim=action_dim,
    num_layers=2,
    seed=SEED
)

dice = Dice(
    q_function=q_func,
    w_function=w_func,
    gamma=0.99,
    q_lr=0.0001,
    w_lr=0.0001,
    lambda_lr=0.0001,
    f1_function=DiceFunctions.DUAL_DICE_P_3_2,
    f2_function=DiceFunctions.CHI_SQUARED,
    method_name='dual_dice',
    seed=SEED,
    device=DEVICE
)

In [5]:
dice.fit(
    state=states,
    action=actions,
    reward=rewards,
    target_action=dqn_actions,
    num_steps=15000,
    batch_size=2048,
    eval_iter=100,
    result_folder='dual_dice_15k_dqn'
)

loss: -0.0707; value: -0.8877: 100%|██████████| 15000/15000 [02:03<00:00, 121.62it/s]


In [6]:
step_reward = dice.predict_per_step_reward(
    state=states, action=actions, reward=rewards
)

traj_reward = dice.predict_per_traj_reward(
    state=states, action=actions, reward=rewards
)

print(f"per step reward: {step_reward:.7f}")
print(f"per trajectory reward: {traj_reward:.7f}")

per step reward: -0.8877100
per trajectory reward: -136.0817834


In [7]:
w = dice.predict_weights(np.concatenate(states), np.concatenate(actions))

w.min(),w.mean(), w.max()

(np.float32(-0.12692545), np.float32(0.88771003), np.float32(1.4718791))